<a href="https://colab.research.google.com/github/Thato20-M/democratic-funnel/blob/main/notebooks/04_census.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Census 2022: eligible adults and context

**Owner:** D  
**Goal:** eligible_adults (citizens 18+), eligible_youth, and socio-economic rates per municipality

**Reads:** `data/raw/census/`  
**Writes:** `INTERIM / "census_2022.csv"`

Run the two setup cells below first, every time.

## Setup

In [3]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo: /content/democratic-funnel


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from src import config
from src.config import RAW, INTERIM, PROCESSED, MODELS, FIGURES, SEED
from src.io_utils import read_any, clean_columns, inventory
from src.privacy import drop_personal

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
print(config.describe())

Environment: Colab
Repo root:   /content/democratic-funnel
Work root:   /content/drive/MyDrive/DIRISA_SDC


## Inspect the tables

Stats SA tables often need a header row and have merged cells.

In [5]:
inventory(RAW / "census")

,file,sheet,size_kb,header_row,rows,n_cols,columns
0,Census2022sample_F18.csv,None,31229.6,0,1338295,5,QID | Province | District | Municipality | Geo...
1,Census2022sample_F19.csv,None,115318.5,0,1338295,32,QID | DERH_HSIZE | DERH_HHSEX | DERH_HHPOP | D...
2,Census2022sample_F21.csv,None,622838.9,0,4230792,46,QID | PID | P02_SEX | P03_YEAR | P03_MONTH | A...


## Eligible population

CITIZENS aged 18+, not all adults. Use the adjusted figures and note the choice.

In [ ]:
# cen["eligible_adults"] = ...
# cen["eligible_youth"] = ...

## Socio-economic variables as rates

Unemployment rate, education, piped water, electricity, refuse removal — shares, not counts.

In [ ]:
# cen["unemployment_rate"] = cen["unemployed"] / cen["labour_force"]

## Check

Provincial totals should match Stats SA's published population figures.

In [ ]:
# print(cen.groupby("province")["eligible_adults"].sum())

## Save output

In [ ]:
# out.to_csv(INTERIM / "census_2022.csv", index=False)
# print(out.shape); out.head()

Load raw files

In [7]:
persons = read_any(RAW / "census" / "Census2022sample_F21.csv")
geo = read_any(RAW / "census" / "Census2022sample_F18.csv")

print(persons.shape)
print(geo.shape)

(4230792, 46)
(1338295, 5)


Clean column names, then confirm real names

In [8]:
persons = clean_columns(persons)
geo = clean_columns(geo)

print(persons.columns.tolist())
print(geo.columns.tolist())

['qid', 'pid', 'p02_sex', 'p03_year', 'p03_month', 'age_group', 'p04_age', 'p05_relation', 'p06_marital_st', 'p07a_pop_group', 'p08_language', 'p09_religious_affiliation', 'p10_citizenship', 'p11_prov_pob', 'p12a_countryofbirth', 'derp_birth_region', 'p12b_yearmoved', 'p13a_usualres', 'p14_since2011', 'p15a_resmonthmoved', 'p15b_resyearmoved', 'derp_prevresprov', 'derp_prev_munic', 'derp_usualresprov', 'derp_usualres_munic', 'p16_main_reason', 'p17a_seeing', 'p17b_hearing', 'p17c_communication', 'p17d_walking', 'p17e_remembering', 'p17f_self_care', 'p17a_devmedeyeglas', 'p17b_devmedhearingaid', 'p17c_devmewalkingstick', 'p17e_prosthesis', 'p17d_devmedwheelchair', 'p17f_devmed_other', 'disability_status', 'p18a_motheralive', 'p18b_fatheralive', 'p19_ecd_attendance', 'p20_eduinst', 'p21_edulevel', 'p22_edufield', 'pers_wgt']
['qid', 'province', 'district', 'municipality', 'geo_type']


Merge persons + geography on shared household ID (qid)

In [9]:
cen = persons.merge(geo, on="qid", how="left")

print(cen.shape)
cen.head()

(4230792, 50)


,qid,pid,p02_sex,p03_year,p03_month,age_group,p04_age,p05_relation,p06_marital_st,p07a_pop_group,p08_language,p09_religious_affiliation,p10_citizenship,p11_prov_pob,p12a_countryofbirth,derp_birth_region,p12b_yearmoved,p13a_usualres,p14_since2011,p15a_resmonthmoved,p15b_resyearmoved,derp_prevresprov,derp_prev_munic,derp_usualresprov,derp_usualres_munic,p16_main_reason,p17a_seeing,p17b_hearing,p17c_communication,p17d_walking,p17e_remembering,p17f_self_care,p17a_devmedeyeglas,p17b_devmedhearingaid,p17c_devmewalkingstick,p17e_prosthesis,p17d_devmedwheelchair,p17f_devmed_other,disability_status,p18a_motheralive,p18b_fatheralive,p19_ecd_attendance,p20_eduinst,p21_edulevel,p22_edufield,pers_wgt,province,district,municipality,geo_type
0,10000003,10000003001,2,1977,8,9,44,1,6,1,4,3,101,2,888,1,8888,1,1,88,8888,1,CPT,1,CPT,88,2,1,1,1,1,1,2,2,2,1,2,2,0,2,2,8,1,9,88,21.567356,1,CPT,CPT,1
1,10000003,10000003002,2,2004,8,4,17,3,6,1,4,3,101,1,888,1,8888,1,1,88,8888,1,CPT,1,CPT,88,2,1,1,1,1,1,2,2,2,2,2,2,0,1,1,8,4,10,88,18.902319,1,CPT,CPT,1
2,10000005,10000005001,2,1980,99,9,41,1,1,1,4,1,101,1,888,1,8888,1,1,88,8888,1,CPT,1,CPT,88,1,1,1,1,1,1,1,2,2,9,9,2,0,2,2,8,1,24,10,21.567356,1,CPT,CPT,1
3,10000005,10000005002,1,1978,99,9,43,2,1,1,4,1,101,2,888,1,8888,1,1,88,8888,1,CPT,1,CPT,88,1,1,1,1,1,1,2,2,2,9,9,2,0,2,2,8,6,28,26,23.281103,1,CPT,CPT,1
4,10000005,10000005003,1,2008,99,3,13,3,6,1,4,1,101,1,888,1,8888,1,1,88,8888,1,CPT,1,CPT,88,1,1,1,1,1,1,2,2,2,9,9,2,0,1,1,8,4,7,88,18.902504,1,CPT,CPT,1


Eligible population

In [10]:
cen["eligible_adults"] = np.where(
    (cen["p10_citizenship"] == 101) & (cen["p04_age"] >= 18),
    cen["pers_wgt"], 0
)

cen["eligible_youth"] = np.where(
    (cen["p10_citizenship"] == 101) & (cen["p04_age"] >= 18) & (cen["p04_age"] <= 34),
    cen["pers_wgt"], 0
)

print(cen[["p10_citizenship", "p04_age", "pers_wgt", "eligible_adults", "eligible_youth"]].head(10))

   p10_citizenship  p04_age   pers_wgt  eligible_adults  eligible_youth
0              101       44  21.567356        21.567356             0.0
1              101       17  18.902319         0.000000             0.0
2              101       41  21.567356        21.567356             0.0
3              101       43  23.281103        23.281103             0.0
4              101       13  18.902504         0.000000             0.0
5              101        3  23.672987         0.000000             0.0
6              101       76  17.958410        17.958410             0.0
7              101       76  18.106597        18.106597             0.0
8              101       50  17.593930        17.593930             0.0
9              101       37   0.335749         0.335749             0.0


Sanity check — provincial totals

In [11]:
pd.set_option("display.float_format", "{:,.0f}".format)
print(cen.groupby("province")["eligible_adults"].sum())

province
1   5,008,537
2   4,546,528
3     879,303
4   1,926,909
5   8,114,913
6   2,395,449
7   9,939,450
8   3,288,050
9   3,951,139
Name: eligible_adults, dtype: float64


Aggregate to municipality level and save

In [13]:
out = cen.groupby("municipality").agg(
    eligible_adults=("eligible_adults", "sum"),
    eligible_youth=("eligible_youth", "sum"),
).reset_index()

out.to_csv(INTERIM / "census_2022.csv", index=False)
print(out.shape)
out.head()

(213, 3)


,municipality,eligible_adults,eligible_youth
0,BUF,"656,889","253,404"
1,CPT,"3,189,834","1,305,385"
2,EC101,"64,962","24,790"
3,EC102,"34,709","12,520"
4,EC104,"68,026","25,557"


In [14]:
# ============================================
# CHECK: Confirm the rebuilt notebook matches the saved file
# ============================================
check = read_any(INTERIM / "census_2022.csv")
print(check.shape)
check.head()

(213, 3)


,municipality,eligible_adults,eligible_youth
0,BUF,"656,889","253,404"
1,CPT,"3,189,834","1,305,385"
2,EC101,"64,962","24,790"
3,EC102,"34,709","12,520"
4,EC104,"68,026","25,557"
